# ROGII: Prefix-Only GR Calibration Audit

A typewell GR curve and a horizontal-well GR log may describe the same geological sequence on different measurement scales. Feeding their raw difference directly into a particle filter, HMM, DTW cost, or cross-correlation can confuse tool gain/offset mismatch with geological mismatch.

This notebook audits a robust affine calibration on the visible prefix of all 773 training wells. Expected GR is obtained by interpolating the paired typewell at known `TVT_input`; observed GR comes from the horizontal well. We fit `observed_GR = gain * typewell_GR + offset` with iterative MAD outlier rejection, then compare raw and calibrated residuals.

Everything here is inference-safe: only the visible prefix and paired typewell are used. No suffix TVT label is read, no submission is created, and no leaderboard feedback is involved.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_CANDIDATES = [
    Path('/kaggle/input/competitions/rogii-wellbore-geology-prediction'),
    Path('/kaggle/input/rogii-wellbore-geology-prediction'),
    Path('data/raw/competition'),
    Path('../../data/raw/competition'),
]
DATA_ROOT = next((path for path in DATA_CANDIDATES if path.exists()), None)
if DATA_ROOT is None:
    raise FileNotFoundError('Competition data was not found')
TRAIN_DIR = DATA_ROOT / 'train'
paths = sorted(TRAIN_DIR.glob('*__horizontal_well.csv'))
print('Data root:', DATA_ROOT)
print('Horizontal wells:', len(paths))

## Robust affine calibration

The fit alternates between a least-squares gain on retained rows, a median residual offset, and a 3.5-MAD inlier mask. Gain is guarded to `[0.35, 2.5]`. The procedure stops after three passes, when the mask stabilizes, or when too few rows remain.

This is intentionally small and transparent. It corrects first-order measurement mismatch; it does not warp TVT, reconstruct missing GR, or choose a geological branch.

In [ ]:
def visible_boundary(values):
    observed = values.notna().to_numpy()
    missing = np.flatnonzero(~observed)
    if len(missing) == 0 or missing[0] < 2 or observed[missing[0]:].any():
        raise ValueError('Expected one finite prefix and one contiguous missing suffix')
    return int(missing[0])

def robust_affine(expected, observed):
    finite = np.isfinite(expected) & np.isfinite(observed)
    x = expected[finite]
    y = observed[finite]
    if len(x) < 20:
        return 1.0, 0.0, np.ones(len(x), dtype=bool)
    keep = np.ones(len(x), dtype=bool)
    gain, offset = 1.0, 0.0
    for _ in range(3):
        design = np.column_stack((x[keep], np.ones(int(keep.sum()))))
        gain, offset = np.linalg.lstsq(design, y[keep], rcond=None)[0]
        gain = float(np.clip(gain, 0.35, 2.5))
        offset = float(np.median(y[keep] - gain * x[keep]))
        residual = y - (gain * x + offset)
        center = float(np.median(residual[keep]))
        mad = float(1.4826 * np.median(np.abs(residual[keep] - center)))
        if mad <= 1e-8:
            break
        new_keep = np.abs(residual - center) <= 3.5 * mad
        if new_keep.sum() < 20 or np.array_equal(new_keep, keep):
            break
        keep = new_keep
    return gain, offset, keep

def robust_sigma(values):
    center = float(np.median(values))
    return float(1.4826 * np.median(np.abs(values - center)))

In [ ]:
records = []
for path in paths:
    well_id = path.name.removesuffix('__horizontal_well.csv')
    horizontal = pd.read_csv(path, usecols=['GR', 'TVT_input'])
    boundary = visible_boundary(horizontal['TVT_input'])
    typewell = pd.read_csv(
        path.with_name(f'{well_id}__typewell.csv'), usecols=['TVT', 'GR'],
    ).dropna().sort_values('TVT').drop_duplicates('TVT')
    known_tvt = horizontal['TVT_input'].to_numpy(dtype=float)[:boundary]
    observed_gr = horizontal['GR'].to_numpy(dtype=float)[:boundary]
    expected_gr = np.interp(
        known_tvt,
        typewell['TVT'].to_numpy(dtype=float),
        typewell['GR'].to_numpy(dtype=float),
    )
    finite = np.isfinite(expected_gr) & np.isfinite(observed_gr)
    x = expected_gr[finite]
    y = observed_gr[finite]
    gain, offset, keep = robust_affine(expected_gr, observed_gr)
    raw_residual = y - x
    calibrated_residual = y - (gain * x + offset)
    correlation = float(np.corrcoef(x, y)[0, 1]) if len(x) > 1 else np.nan
    records.append({
        'well_id': well_id,
        'prefix_rows': boundary,
        'finite_pairs': len(x),
        'gain': gain,
        'offset': offset,
        'prefix_correlation': correlation,
        'retained_fraction': float(keep.mean()),
        'raw_rmse': float(np.sqrt(np.mean(raw_residual**2))),
        'calibrated_rmse': float(np.sqrt(np.mean(calibrated_residual**2))),
        'raw_robust_sigma': robust_sigma(raw_residual),
        'calibrated_robust_sigma': robust_sigma(calibrated_residual),
    })

audit = pd.DataFrame(records)
audit['rmse_ratio'] = audit['calibrated_rmse'] / audit['raw_rmse']
audit['robust_sigma_ratio'] = audit['calibrated_robust_sigma'] / audit['raw_robust_sigma']
display(audit.head())

In [ ]:
summary = pd.Series({
    'wells': len(audit),
    'median gain': audit['gain'].median(),
    'median offset': audit['offset'].median(),
    'median prefix correlation': audit['prefix_correlation'].median(),
    'median retained fraction': audit['retained_fraction'].median(),
    'median calibrated/raw RMSE ratio': audit['rmse_ratio'].median(),
    'median calibrated/raw robust-sigma ratio': audit['robust_sigma_ratio'].median(),
    'fraction with lower RMSE': (audit['rmse_ratio'] < 1.0).mean(),
    'fraction with lower robust sigma': (audit['robust_sigma_ratio'] < 1.0).mean(),
    'fraction at a gain guard': (
        np.isclose(audit['gain'], 0.35) | np.isclose(audit['gain'], 2.5)
    ).mean(),
})
display(summary.to_frame('value').style.format('{:.4f}'))
display(audit[[
    'gain', 'offset', 'prefix_correlation', 'retained_fraction',
    'raw_rmse', 'calibrated_rmse', 'raw_robust_sigma', 'calibrated_robust_sigma',
]].describe(percentiles=[0.05, 0.10, 0.50, 0.90, 0.95]).T)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes[0, 0].hist(audit['gain'], bins=35, color='#2a6fbb', alpha=0.85)
axes[0, 0].axvline(1.0, color='black', linestyle='--', linewidth=1)
axes[0, 0].set(title='Prefix affine gain', xlabel='gain', ylabel='wells')
axes[0, 1].hist(audit['offset'], bins=35, color='#347a4a', alpha=0.85)
axes[0, 1].axvline(0.0, color='black', linestyle='--', linewidth=1)
axes[0, 1].set(title='Prefix affine offset', xlabel='offset', ylabel='wells')
axes[1, 0].scatter(audit['raw_rmse'], audit['calibrated_rmse'], s=13, alpha=0.5)
limit = max(audit['raw_rmse'].max(), audit['calibrated_rmse'].max())
axes[1, 0].plot([0, limit], [0, limit], 'k--', linewidth=1)
axes[1, 0].set(title='Residual RMSE before and after calibration', xlabel='raw RMSE', ylabel='calibrated RMSE')
axes[1, 1].scatter(audit['prefix_correlation'], audit['calibrated_robust_sigma'], s=13, alpha=0.5, color='#c45a32')
axes[1, 1].set(title='Shape agreement versus remaining noise', xlabel='prefix correlation', ylabel='calibrated robust sigma')
plt.tight_layout()
plt.show()

## Best and worst prefix agreements

A low calibrated residual is a cleaner likelihood input. A high residual after calibration indicates that gain and offset are not the main problem: the curves may be locally warped, incomplete, noisy, or genuinely inconsistent. It is not valid to force the likelihood to be sharp in those wells.

In [ ]:
selected = pd.concat([
    audit.nsmallest(2, 'calibrated_rmse'),
    audit.nlargest(2, 'calibrated_rmse'),
]).drop_duplicates('well_id')
fig, axes = plt.subplots(len(selected), 1, figsize=(13, 3 * len(selected)), squeeze=False)
for axis, (_, record) in zip(axes[:, 0], selected.iterrows()):
    path = TRAIN_DIR / f'{record.well_id}__horizontal_well.csv'
    horizontal = pd.read_csv(path, usecols=['GR', 'TVT_input'])
    boundary = visible_boundary(horizontal['TVT_input'])
    typewell = pd.read_csv(
        path.with_name(f'{record.well_id}__typewell.csv'), usecols=['TVT', 'GR'],
    ).dropna().sort_values('TVT').drop_duplicates('TVT')
    known_tvt = horizontal['TVT_input'].to_numpy(dtype=float)[:boundary]
    observed = horizontal['GR'].to_numpy(dtype=float)[:boundary]
    expected = np.interp(known_tvt, typewell['TVT'], typewell['GR'])
    calibrated = record.gain * expected + record.offset
    thin = max(1, len(known_tvt) // 800)
    axis.plot(known_tvt[::thin], observed[::thin], label='horizontal observed', linewidth=1.0)
    axis.plot(known_tvt[::thin], expected[::thin], label='raw typewell', linewidth=0.9, alpha=0.75)
    axis.plot(known_tvt[::thin], calibrated[::thin], label='calibrated typewell', linewidth=0.9, alpha=0.85)
    axis.set(
        title=(
            f'{record.well_id}: corr={record.prefix_correlation:.3f}, '
            f'raw/cal RMSE={record.raw_rmse:.1f}/{record.calibrated_rmse:.1f}'
        ),
        xlabel='known prefix TVT', ylabel='GR',
    )
    axis.legend(ncol=3)
plt.tight_layout()
plt.show()

display(audit.nlargest(15, 'calibrated_rmse')[[
    'well_id', 'gain', 'offset', 'prefix_correlation', 'retained_fraction',
    'raw_rmse', 'calibrated_rmse', 'calibrated_robust_sigma',
]])

## Modeling implications

- Fit GR scale only on the visible prefix. Using suffix TVT to calibrate test likelihoods would be target leakage.
- Robust affine calibration is a measurement-model correction, not a geological alignment algorithm.
- Carry the calibrated residual scale into PF/HMM likelihoods; a single global GR sigma overstates confidence on difficult wells.
- A poor prefix correlation or large residual can justify a wider likelihood, but it is not by itself a proven fallback gate.
- Report gain guards and retained fraction. Silent clipping or heavy rejection is a diagnostic event.
- Combine calibrated GR evidence with trajectory continuity, multiple branches, and an anchor prior.

Calibrate the sensor mismatch first. Then ask the geological question.